# Update Newspaper Endorsements Archive

This notebook updates the Endorsement Archive from `RA_checked/` and `done/` folders.

**Priority logic:**
1. `RA_checked/` papers overwrite whatever is in the archive (RA-finalized data wins)
2. `done/` papers are copied ONLY if they don't already exist in the archive
3. Manifest and README stats are regenerated

**Expected directory layout:**
```
newspaper endorsement processing/
  done/                        <- QA'd extraction output
  RA_checked/                  <- RA-reviewed and corrected data
  newspaper_endorsements/      <- this repo
    Endorsement Archive/
    Processing Code/           <- this notebook lives here
    ...
```

## Setup

In [ ]:
import csv
import os
import re
import shutil
from datetime import date

# ── Locate directories ──────────────────────────────────────────

# Find repo root (parent of Processing Code/)
notebook_dir = os.path.abspath('')
if os.path.basename(notebook_dir) == 'Processing Code':
    repo_root = os.path.dirname(notebook_dir)
else:
    repo_root = notebook_dir

archive_dir = os.path.join(repo_root, 'Endorsement Archive')
processing_dir = os.path.join(repo_root, 'Processing Code')

# done/ and RA_checked/ are siblings of the repo folder
workspace = os.path.dirname(repo_root)
done_dir = os.path.join(workspace, 'done')
ra_dir = os.path.join(workspace, 'RA_checked')

# Validate
assert os.path.isdir(archive_dir), f'Endorsement Archive not found at {archive_dir}'
print(f'Repo root:    {repo_root}')
print(f'Archive:      {archive_dir}')
print(f'done/:        {done_dir} {"(exists)" if os.path.isdir(done_dir) else "(NOT FOUND)"}')
print(f'RA_checked/:  {ra_dir} {"(exists)" if os.path.isdir(ra_dir) else "(NOT FOUND)"}')

In [ ]:
# ── State mapping ──────────────────────────────────────────────

STATE_NAMES = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
    'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
    'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
    'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
    'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
    'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia',
}


def get_state_from_folder(folder_path):
    """Read the first CSV row to determine the newspaper's state."""
    for fn in os.listdir(folder_path):
        if fn.endswith('_candidates.csv') or fn.endswith('_propositions.csv'):
            filepath = os.path.join(folder_path, fn)
            try:
                with open(filepath) as f:
                    reader = csv.DictReader(f)
                    for row in reader:
                        abbr = row.get('state_newspaper', '').strip()
                        if abbr in STATE_NAMES:
                            return STATE_NAMES[abbr]
            except Exception:
                pass
    return None


def find_paper_in_archive(paper_name, archive_dir):
    """Check if a paper already exists anywhere in the archive."""
    for state_folder in os.listdir(archive_dir):
        state_path = os.path.join(archive_dir, state_folder)
        if not os.path.isdir(state_path):
            continue
        paper_path = os.path.join(state_path, paper_name)
        if os.path.isdir(paper_path):
            return paper_path
    return None


def copy_paper(src_dir, dest_dir):
    """Copy a paper folder, overwriting if it exists."""
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    shutil.copytree(src_dir, dest_dir)


print('Helper functions loaded.')

## Step 1: Copy from RA_checked/ (highest priority)

RA-finalized papers overwrite whatever version is currently in the archive.

In [ ]:
ra_copied = []

if os.path.isdir(ra_dir):
    for paper in sorted(os.listdir(ra_dir)):
        src = os.path.join(ra_dir, paper)
        if not os.path.isdir(src):
            continue
        if not any(f.endswith('.csv') for f in os.listdir(src)):
            print(f'  SKIP {paper}: no CSV files')
            continue

        state = get_state_from_folder(src)
        if not state:
            print(f'  SKIP {paper}: cannot determine state')
            continue

        dest_state = os.path.join(archive_dir, state)
        os.makedirs(dest_state, exist_ok=True)
        dest = os.path.join(dest_state, paper)

        existed = os.path.exists(dest)
        copy_paper(src, dest)
        action = 'UPDATED' if existed else 'ADDED'
        print(f'  {action} {state}/{paper} (from RA_checked)')
        ra_copied.append(paper)
else:
    print('  RA_checked/ not found, skipping')

print(f'\nTotal from RA_checked: {len(ra_copied)}')

## Step 2: Copy from done/ (fill gaps)

Papers from `done/` are copied only if they don't already exist in the archive (i.e., they weren't already copied from `RA_checked/` or a previous run).

In [ ]:
done_copied = []
done_skipped = []

if os.path.isdir(done_dir):
    for paper in sorted(os.listdir(done_dir)):
        src = os.path.join(done_dir, paper)
        if not os.path.isdir(src):
            continue
        if not any(f.endswith('.csv') for f in os.listdir(src)):
            continue

        existing = find_paper_in_archive(paper, archive_dir)
        if existing:
            done_skipped.append(paper)
            continue

        state = get_state_from_folder(src)
        if not state:
            print(f'  SKIP {paper}: cannot determine state')
            continue

        dest_state = os.path.join(archive_dir, state)
        os.makedirs(dest_state, exist_ok=True)
        dest = os.path.join(dest_state, paper)

        copy_paper(src, dest)
        print(f'  ADDED {state}/{paper} (from done/)')
        done_copied.append(paper)
else:
    print('  done/ not found, skipping')

print(f'\nAdded from done/: {len(done_copied)}')
print(f'Already in archive (skipped): {len(done_skipped)}')

## Step 3: Regenerate qa_manifest.csv

Scans the archive and rebuilds the manifest with correct paths, QA dates, and results.

In [ ]:
manifest_rows = []

for state_folder in sorted(os.listdir(archive_dir)):
    state_path = os.path.join(archive_dir, state_folder)
    if not os.path.isdir(state_path):
        continue
    for paper in sorted(os.listdir(state_path)):
        paper_path = os.path.join(state_path, paper)
        if not os.path.isdir(paper_path):
            continue

        files = os.listdir(paper_path)
        cand_csv = next((f for f in files if f.endswith('_candidates.csv')), None)
        prop_csv = next((f for f in files if f.endswith('_propositions.csv')), None)

        if not cand_csv and not prop_csv:
            continue

        cand_rel = os.path.join('Endorsement Archive', state_folder, paper, cand_csv) if cand_csv else ''
        prop_rel = os.path.join('Endorsement Archive', state_folder, paper, prop_csv) if prop_csv else ''

        # Check QA report for status
        qa_report = os.path.join(paper_path, 'QA_REPORT.md')
        qa_result = 'PASS WITH FIXES'
        qa_date = str(date.today())

        if os.path.exists(qa_report):
            with open(qa_report) as f:
                content = f.read()
            date_match = re.search(r'\*\*Audit date:\*\*\s*(\S+)', content)
            if date_match:
                qa_date = date_match.group(1)
            result_match = re.search(r'## Overall Assessment\s+(\S.*)', content)
            if result_match:
                qa_result = result_match.group(1).strip()

        manifest_rows.append({
            'folder_name': paper,
            'candidates_csv': cand_rel,
            'propositions_csv': prop_rel,
            'qa_date': qa_date,
            'qa_result': qa_result,
        })

manifest_path = os.path.join(processing_dir, 'qa_manifest.csv')
with open(manifest_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['folder_name', 'candidates_csv', 'propositions_csv', 'qa_date', 'qa_result'])
    writer.writeheader()
    writer.writerows(manifest_rows)

print(f'Wrote {len(manifest_rows)} entries to qa_manifest.csv')
for r in manifest_rows:
    print(f"  {r['folder_name']:35s} {r['qa_result']}")

## Step 4: Update COVERAGE.md and README.md

Writes the full per-newspaper table to `COVERAGE.md` and updates the compact summary + abstract stats in `README.md`.

In [ ]:
stats = []
total_cand = 0
total_prop = 0
all_years = set()
state_stats = {}  # state -> {papers, cand, prop, years}

for state_folder in sorted(os.listdir(archive_dir)):
    state_path = os.path.join(archive_dir, state_folder)
    if not os.path.isdir(state_path):
        continue

    if state_folder not in state_stats:
        state_stats[state_folder] = {'papers': 0, 'cand': 0, 'prop': 0, 'years': set()}

    for paper in sorted(os.listdir(state_path)):
        paper_path = os.path.join(state_path, paper)
        if not os.path.isdir(paper_path):
            continue

        cand_count = 0
        prop_count = 0
        paper_years = set()
        newspaper_id = ''

        for fn in os.listdir(paper_path):
            if fn.endswith('_candidates.csv'):
                with open(os.path.join(paper_path, fn)) as f:
                    rows = list(csv.DictReader(f))
                    cand_count = len(rows)
                    for r in rows:
                        y = r.get('year', '').strip()
                        if y:
                            paper_years.add(int(y))
                            all_years.add(int(y))
                    if rows:
                        newspaper_id = rows[0].get('newspaper_id', '')
            elif fn.endswith('_propositions.csv'):
                with open(os.path.join(paper_path, fn)) as f:
                    rows = list(csv.DictReader(f))
                    prop_count = len(rows)
                    for r in rows:
                        y = r.get('year', '').strip()
                        if y:
                            paper_years.add(int(y))
                            all_years.add(int(y))
                    if rows and not newspaper_id:
                        newspaper_id = rows[0].get('newspaper_id', '')

        if cand_count == 0 and prop_count == 0:
            continue

        total_cand += cand_count
        total_prop += prop_count
        yr_range = f'{min(paper_years)}-{max(paper_years)}' if paper_years else 'N/A'
        nid_display = newspaper_id if newspaper_id else '--'

        stats.append((state_folder, paper, nid_display, cand_count, prop_count, yr_range))
        state_stats[state_folder]['papers'] += 1
        state_stats[state_folder]['cand'] += cand_count
        state_stats[state_folder]['prop'] += prop_count
        state_stats[state_folder]['years'].update(paper_years)

# Remove states with no data
state_stats = {k: v for k, v in state_stats.items() if v['papers'] > 0}

print(f'Archive contains {len(stats)} newspapers across {len(state_stats)} states')
print(f'Total: {total_cand:,} candidates + {total_prop:,} propositions = {total_cand + total_prop:,} records')
print(f'Year range: {min(all_years)}-{max(all_years)}')

In [ ]:
# ── Write COVERAGE.md (full per-newspaper table) ──────────────

paper_table = []
paper_table.append('| State | Newspaper | ID | Candidates | Propositions | Year Range |')
paper_table.append('|-------|-----------|---:|----------:|-----------:|-----------|')
for state, paper, nid, cc, pc, yr in stats:
    paper_table.append(f'| {state} | {paper} | {nid} | {cc:,} | {pc:,} | {yr} |')
paper_table.append(
    f'| **Total** | **{len(stats)} newspapers, {len(state_stats)} states** | | '
    f'**{total_cand:,}** | **{total_prop:,}** | '
    f'**{min(all_years)}-{max(all_years)}** |'
)

state_table = []
state_table.append('| State | Newspapers | Candidates | Propositions | Total | Year Range |')
state_table.append('|-------|----------:|----------:|-----------:|------:|-----------|')
for st in sorted(state_stats):
    s = state_stats[st]
    total = s['cand'] + s['prop']
    yr = f"{min(s['years'])}-{max(s['years'])}" if s['years'] else 'N/A'
    state_table.append(f"| {st} | {s['papers']} | {s['cand']:,} | {s['prop']:,} | {total:,} | {yr} |")

coverage_md = f"""# Newspaper Coverage Detail

Full listing of all newspapers in the Endorsement Archive, organized by state.

*Last updated: {date.today()}*

## Coverage by Newspaper

{chr(10).join(paper_table)}

## Coverage by State

{chr(10).join(state_table)}
"""

coverage_path = os.path.join(repo_root, 'COVERAGE.md')
with open(coverage_path, 'w') as f:
    f.write(coverage_md)
print(f'Wrote COVERAGE.md: {len(stats)} papers')

# ── Update README.md (compact summary only) ───────────────────

readme_path = os.path.join(repo_root, 'README.md')
with open(readme_path) as f:
    content = f.read()

# Build compact state summary table
compact_state = []
compact_state.append('| State | Newspapers | Records | Year Range |')
compact_state.append('|-------|----------:|--------:|-----------|')
for st in sorted(state_stats):
    s = state_stats[st]
    total = s['cand'] + s['prop']
    yr = f"{min(s['years'])}-{max(s['years'])}" if s['years'] else 'N/A'
    compact_state.append(f"| {st} | {s['papers']} | {total:,} | {yr} |")

# Replace the bold summary line
content = re.sub(
    r'\*\*\d+ newspapers\*\* across \*\*\d+ states\*\*, covering elections from \*\*\d+-\d+\*\*\.',
    f'**{len(stats)} newspapers** across **{len(state_stats)} states**, covering elections from **{min(all_years)}-{max(all_years)}**.',
    content
)

# Replace the record count table
content = re.sub(
    r'\| \| Candidates \| Propositions \| Total \|\n\|---\|[-\d:| ]+\|\n\| \*\*Records\*\* \|.*?\|',
    f'| | Candidates | Propositions | Total |\n|---|----------:|-----------:|------:|\n| **Records** | {total_cand:,} | {total_prop:,} | {total_cand + total_prop:,} |',
    content
)

# Replace the state summary table
state_pattern = r'(\| State \| Newspapers \| Records \| Year Range \|\n\|[-|: ]+\n)((?:\|.*\n)*)'
state_match = re.search(state_pattern, content)
if state_match:
    new_state_block = '\n'.join(compact_state) + '\n'
    content = content[:state_match.start()] + new_state_block + content[state_match.end():]

# Update the abstract
content = re.sub(
    r'currently contains over [\d,]+ endorsement records across \d+ newspapers in \d+ states',
    f'currently contains over {total_cand + total_prop:,} endorsement records across {len(stats)} newspapers in {len(state_stats)} states',
    content
)

with open(readme_path, 'w') as f:
    f.write(content)
print(f'Updated README.md compact summary')

## Summary

In [ ]:
print('=' * 60)
print('UPDATE COMPLETE')
print('=' * 60)
print(f'  Papers from RA_checked: {len(ra_copied)}')
print(f'  Papers from done/:      {len(done_copied)}')
print(f'  Already in archive:     {len(done_skipped)}')
print(f'  Total in archive:       {len(stats)}')
print(f'  Total records:          {total_cand + total_prop:,} ({total_cand:,} candidates, {total_prop:,} propositions)')
print(f'  Year range:             {min(all_years)}-{max(all_years)}')
print(f'  States:                 {len(state_count)}')
print()
print('Next steps:')
print('  1. Run compile_all.R to rebuild master datasets')
print('  2. Run augmentation on compiled data')
print('  3. git add / commit / push')